In [ ]:
# Import packages and plotting setup
import matplotlib.pyplot as plt
from pathlib import Path

import numpy as np
import pandas as pd
import mpltern

from Functions import cartesian_to_ternary, vmf_kde, sqrt_transform, setup_matplotlib_parameter

wong_cycle = setup_matplotlib_parameter()

In [ ]:
# Load data
data_dir = Path("..") / "Data files" / "Inclusion lists per stage"

df_container = {}
for file in data_dir.glob("*.xlsx"):
    df_container[file.stem] = pd.read_excel(file, sheet_name="NMI data", index_col=0)

# Merge dataframes from all samples into one
s1_merged = pd.concat([df_container["S1-DO-1"], df_container["S1-DO-2"], df_container["S1-DO-3"], df_container["S1-DO-4"]]).fillna(0)

s2_merged = pd.concat([df_container["S2-DS-1"], df_container["S2-DS-2"], df_container["S2-DS-3"], df_container["S2-DS-4"]]).fillna(0)

s3_merged = pd.concat([df_container["S3-VT-1"], df_container["S3-VT-2"], df_container["S3-VT-3"], df_container["S3-VT-4"]]).fillna(0)

s4_merged = pd.concat([df_container["S4-CT-1"], df_container["S4-CT-2"]]).fillna(0)

s5_merged = pd.concat([df_container["S5-CT-1"], df_container["S5-CT-2"]]).fillna(0)

s6_merged = pd.concat([df_container["S6-CT-1"], df_container["S6-CT-2"]]).fillna(0)

s7_merged = pd.concat([df_container["S7-CT-1"], df_container["S7-CT-2"], df_container["S7-CT-3"], df_container["S7-CT-4"]]).fillna(0)

s8_merged = pd.concat([df_container["S8-TU-1"], df_container["S8-TU-2"], df_container["S8-TU-3"], df_container["S8-TU-4"]]).fillna(0)

In [ ]:
# Ternary grid for KDE
grid_size = 200
x = np.linspace(0, 1, grid_size)
y = np.linspace(0, np.sqrt(3) / 2, grid_size)
xx, yy = np.meshgrid(x, y)

grid_points = cartesian_to_ternary(xx.ravel(), yy.ravel())
valid_mask = np.all(grid_points >= 0, axis=1)
valid_grid = grid_points[valid_mask]

valid_grid_sphere = sqrt_transform(valid_grid)

In [ ]:
# Converstion to oxides
# Molar masses (g/mol)
M_Al  = 26.98
M_Ca  = 40.08
M_Mg  = 24.31
M_S   = 32.06
M_O   = 15.99

M_Al2O3 = 2 * M_Al + 3 * M_O
M_CaO   = M_Ca + M_O
M_MgO   = M_Mg + M_O
M_CaS   = M_Ca + M_S

# Conversion factors (mass of oxide per mass of element)
f_Al2O3 = M_Al2O3 / (2 * M_Al)  # divide by 2 because 2 Al per formula unit
f_MgO   = M_MgO   / M_Mg

In [ ]:
def molar_to_weight(n_CaO, n_Al2O3):
    w_CaO   = n_CaO   * M_CaO
    w_Al2O3 = n_Al2O3 * M_Al2O3
    total   = w_CaO + w_Al2O3
    return w_CaO/total*100, w_Al2O3/total*100

# Add CA phase labels
phases = {
    r'$\mathrm{C_3A}$':   (0, molar_to_weight(3, 1)[0], molar_to_weight(3, 1)[1]),
    r'$\mathrm{C_{12}A_7}$': (0, molar_to_weight(12, 7)[0], molar_to_weight(12, 7)[1]),
    r'$\mathrm{CA}$':    (0, molar_to_weight(1, 1)[0], molar_to_weight(1, 1)[1]),
    r'$\mathrm{CA_2}$':   (0, molar_to_weight(1, 2)[0], molar_to_weight(1, 2)[1]),
    r'$\mathrm{CA_6}$':   (0, molar_to_weight(1, 6)[0], molar_to_weight(1, 6)[1]),
}

In [ ]:
Z_container_S1_3 = []
bandwidths = [0.15, 0.10, 0.1]

for i, merged_dfs in enumerate([s1_merged, s2_merged, s3_merged]):
    X = merged_dfs.loc[:, ["MgO", "Al2O3", "CaO"]].to_numpy()

    # Generate oxidic Al203-CaO-MgO composition
    X_MgO = X[:, 0]
    X_Al2O3 = X[:, 1]
    X_CaO = X[:, 2]

    X_oxidic = np.column_stack((X_CaO, X_Al2O3, X_MgO))

    X_oxidic = X_oxidic[X_oxidic.sum(axis=1) > 0]  # remove rows with sum zero

    # Scale between 0 and 1
    X_oxidic_norm = X_oxidic / X_oxidic.sum(axis=1, keepdims=True)
    Z = vmf_kde(X_oxidic_norm, valid_grid_sphere, bandwidth=bandwidths[i], normalize=True, scaling="linear")
    Z_container_S1_3.append(Z)

In [ ]:
Z_container_oxidic_S4_8 = []
Z_container_sulfidic_S4_8 = []
bandwidths_oxidic = [0.2, 0.2, 0.15, 0.10, 0.1]  # S4, S5, S6, S7
bandwidths_sulfidic = [0.2, 0.2, 0.10, 0.08, 0.1]  # S4, S5, S6, S7
ecd_filter = False

for i, merged_dfs in enumerate([s4_merged, s5_merged, s6_merged, s7_merged, s8_merged]):

    if ecd_filter:
        # ECD filter
        mask_ecd = merged_dfs["ECD (µm)"] >= 1.0

        X = merged_dfs.loc[mask_ecd, ["MgO", "Al2O3", "CaO", "CaS"]].to_numpy()
    else:
        X = merged_dfs.loc[:, ["MgO", "Al2O3", "CaO", "CaS"]].to_numpy()

    # Generate oxidic Al-Ca-Mg composition
    X_MgO = X[:, 0]
    X_Al2O3 = X[:, 1]
    X_CaO = X[:, 2]

    X_oxidic = np.column_stack((X_MgO, X_CaO, X_Al2O3))

    X_oxidic = X_oxidic[X_oxidic.sum(axis=1) > 0]  # remove rows with sum zero

    # Scale between 0 and 1
    X_oxidic_norm = X_oxidic / X_oxidic.sum(axis=1, keepdims=True)

    # Usage
    Z = vmf_kde(X_oxidic_norm, valid_grid_sphere, bandwidth=bandwidths_oxidic[i],
                normalize=True, scaling="linear")
    Z_container_oxidic_S4_8.append(Z)

    # Generate CaS-Al2O3-CaO composition
    X_CaS = X[:, 3]

    X_sulfidic = np.column_stack((X_CaS, X_Al2O3, X_CaO))
    X_sulfidic = X_sulfidic[X_sulfidic.sum(axis=1) > 0]  # remove rows with sum zero

    # Scale between 0 and 1
    X_sulfidic_norm = X_sulfidic / X_sulfidic.sum(axis=1, keepdims=True)

    # Usage
    Z = vmf_kde(X_sulfidic_norm, valid_grid_sphere, bandwidth=bandwidths_sulfidic[i],
                normalize=True, scaling="linear")
    Z_container_sulfidic_S4_8.append(Z)

In [ ]:
import matplotlib as mpl

mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Arial'
mpl.rcParams['mathtext.it'] = 'Arial:italic'
mpl.rcParams['mathtext.bf'] = 'Arial:bold'
mpl.rcParams['mathtext.default'] = 'regular'

# marker and size per phase; sizes compensate for perceived area
phase_style = {
    r"$\mathrm{C_3A}$":   ("o", 40, 1.1, "1"),
    r"$\mathrm{C_{12}A_7}$": ("o", 40, 1.1, "0.8"),
    r"$\mathrm{CA}$":    ("o", 40, 1.1, "0.6"),
    r"$\mathrm{CA_2}$":   ("o", 40, 1.1, "0.4"),
    r"$\mathrm{CA_6}$":   ("o", 40, 1.1, "0.2"),
}

In [ ]:
fig = plt.figure(figsize=(210/25.4, 297/25.4))

outer = fig.add_gridspec(nrows=4, ncols=2, wspace=0.25, hspace=0.5)

ticks = np.arange(0, 101, 20)

# Store axes
axs_single = []
axs_pairs = []

# PLotting options
levels = np.linspace(0, 1, 11)

# First three subfigures: one ternary diagram each
for i in range(3):
    ax = fig.add_subplot(
        outer[i],
        projection='ternary',
        ternary_sum=100
    )
    axs_single.append(ax)

    # your existing setup
    ax.set_llabel('')
    ax.set_rlabel('')
    ax.set_tlabel(r"$\mathrm{CaO}$", fontweight="normal")
    ax.taxis.labelpad = 1

    ax.text(-0.18, -0.1, r'$\mathrm{Al_2O_3}$', transform=ax.transAxes, ha='center', va='center', fontsize=12, weight="normal")
    ax.text(1.18, -0.1, r"$\mathrm{MgO}$", transform=ax.transAxes, ha='center', va='center', fontsize=12, weight="normal")

    ax.laxis.set_label_rotation_mode('horizontal')
    ax.raxis.set_label_rotation_mode('horizontal')
    ax.tick_params(labelrotation='horizontal')

    ax.taxis.set_ticks(ticks)
    ax.raxis.set_ticks(ticks)
    ax.laxis.set_ticks(ticks)

    cs = ax.tricontourf(valid_grid[:, 0], valid_grid[:, 1], valid_grid[:, 2],
                        Z_container_S1_3[i],
                        levels=levels,
                        cmap='Blues')

    ax.tricontour(valid_grid[:, 0], valid_grid[:, 1], valid_grid[:, 2],
                  Z_container_S1_3[i],
                  levels=levels,
                  colors="0.25",
                  linewidths=0.35,)

# Remaining subfigures: two ternary diagrams each
for j in range(5):   # use range(4) if you only have S4-S7
    slot = outer[j + 3]
    inner = slot.subgridspec(nrows=1, ncols=2, wspace=-0.47, hspace=0)
    ax_oxidic = fig.add_subplot(inner[0], projection='ternary', rotation=60, ternary_sum=100)
    ax_sulfidic = fig.add_subplot(inner[1], projection='ternary', rotation=-120, ternary_sum=100)

    ax_oxidic.set_zorder(ax_sulfidic.get_zorder() + 1)
    ax_oxidic.patch.set_alpha(0)
    axs_pairs.append((ax_oxidic, ax_sulfidic))



for i, (ax_oxidic, ax_sulfidic) in enumerate(axs_pairs):
    # oxidic CaO-Al2O3-MgO system
    ax_oxidic.text(-0.17, 1.12, r"$\mathrm{MgO}$", transform=ax_oxidic.transAxes, ha='center', va='center',
                   fontsize=12, weight='normal')
    ax_oxidic.text(1.25, 1.12, r'$\mathrm{Al_2O_3}$', transform=ax_oxidic.transAxes, ha='center', va='center',
                   fontsize=12)
    ax_oxidic.text(0.37, -0.12, r"$\mathrm{CaO}$", transform=ax_oxidic.transAxes, ha='center', va='center',
                   fontsize=12, weight='normal')

    ax_oxidic.laxis.set_ticks(ticks)
    ax_oxidic.taxis.set_ticks(ticks)
    ax_oxidic.raxis.set_ticks(ticks)

    ax_oxidic.laxis.set_ticklabels(ticks)
    ax_oxidic.raxis.set_ticklabels([])
    ax_oxidic.raxis.set_tick_params(
        tick1On=False,
        tick2On=False,
        label1On=False,
        label2On=False
    )

    ax_oxidic.taxis.set_label_rotation_mode('horizontal')
    ax_oxidic.laxis.set_label_rotation_mode('horizontal')
    ax_oxidic.raxis.set_label_rotation_mode('horizontal')
    ax_oxidic.tick_params(labelrotation='horizontal')

    cs = ax_oxidic.tricontourf(
        valid_grid[:, 0],
        valid_grid[:, 1],
        valid_grid[:, 2],
        Z_container_oxidic_S4_8[i],
        levels=levels,
        cmap='Blues'
    )

    ax_oxidic.tricontour(
        valid_grid[:, 0],
        valid_grid[:, 1],
        valid_grid[:, 2],
        Z_container_oxidic_S4_8[i],
        levels=levels,
        colors="0.25",
        linewidths=0.35,
    )

    # sulfidic CaS-Al2O3-CaO system
    ax_sulfidic.text(1.18, -0.12, r"$\mathrm{CaS}$",
                     transform=ax_sulfidic.transAxes, ha='center', va='center',
                     fontsize=12, weight='normal')

    ax_sulfidic.laxis.set_ticks(ticks)
    ax_sulfidic.taxis.set_ticks(ticks)
    ax_sulfidic.raxis.set_ticks(ticks)

    ax_sulfidic.laxis.set_ticklabels([str(t) for t in np.flip(ticks)])
    ax_sulfidic.raxis.set_ticklabels([])
    ax_sulfidic.taxis.set_ticklabels([str(t) for t in np.flip(ticks)])

    ax_sulfidic.raxis.set_tick_params(tick1On=False, tick2On=True)
    ax_sulfidic.taxis.set_tick_params(tick1On=False, tick2On=True)
    ax_sulfidic.laxis.set_tick_params(tick1On=False, tick2On=False)

    ax_sulfidic.taxis.set_label_rotation_mode('horizontal')
    ax_sulfidic.laxis.set_label_rotation_mode('horizontal')
    ax_sulfidic.raxis.set_label_rotation_mode('horizontal')
    ax_sulfidic.tick_params(labelrotation='horizontal')

    ax_sulfidic.tricontourf(
        valid_grid[:, 0],
        valid_grid[:, 1],
        valid_grid[:, 2],
        Z_container_sulfidic_S4_8[i],
        levels=levels,
        cmap='Blues'
    )

    ax_sulfidic.tricontour(
        valid_grid[:, 0],
        valid_grid[:, 1],
        valid_grid[:, 2],
        Z_container_sulfidic_S4_8[i],
        levels=levels,
        # colors="k",
        colors="0.25",
        linewidths=0.35,
        # linewidths=0.5
    )

    if i == 4:
        # Add phase markers
        for name, (t, l, r) in phases.items():
            marker, size, lw, edgecolor = phase_style[name]
            ax_oxidic.scatter(
                t-1.3, l, r,
                s=size,
                marker=marker,
                facecolors=edgecolor,
                edgecolors="0.15",
                linewidths=lw,
                zorder=100,
                clip_on=False
            )


import string

fig_labels = [f"({letter})" for letter in string.ascii_lowercase[:8]]

for label, ax in zip(fig_labels[:3], axs_single):
    ax.text(
        -0.60, 1.29,
        label,
        transform=ax.transAxes,
        fontsize=12,
        fontweight='bold',
        ha='left',
        va='top'
    )

for label, (ax_oxidic, ax_sulfidic) in zip(fig_labels[3:], axs_pairs):
    ax_oxidic.text(
        -0.35, 1.3,
        label,
        transform=ax_oxidic.transAxes,
        fontsize=12,
        fontweight='bold',
        ha='left',
        va='top'
    )

fig.tight_layout()

# Add legends
from matplotlib.lines import Line2D

iso_liquidus_handle_100 = Line2D([0], [0], color="darkorange", linewidth=1.2,
                                 label="100 %")

iso_liquidus_handle_20 = Line2D([0], [0], color="darkorange", linewidth=1.2, linestyle="--",
                                 label="20 %")

# Legend
phase_handles = [
    Line2D([0], [0], linestyle="None", marker=phase_style[n][0],
           markerfacecolor=phase_style[n][3], markeredgecolor="0.15",
           markeredgewidth=1.1, markersize=np.sqrt(phase_style[n][1]),
           label=n)
    for i, n in reversed(list(enumerate(phases)))
]

phase_legend = axs_pairs[-1][1].legend(
    handles=phase_handles,
    loc="upper left", bbox_to_anchor=(.99, 0.9),
    frameon=False, handlelength=1.0, handletextpad=0.5,
    labelspacing=0.35,
    title_fontsize="small", fontsize="small",
)

import matplotlib.transforms as transforms

fig.canvas.draw()

offset_l = transforms.ScaledTranslation(-3/72, 9/72, fig.dpi_scale_trans)
offset_t = transforms.ScaledTranslation(8/72, 0/72, fig.dpi_scale_trans)

for ax_oxidic, ax_sulfidic in axs_pairs:
    for tick in ax_sulfidic.laxis.get_major_ticks():
        tick.label1.set_transform(tick.label1.get_transform() + offset_l)

    for tick in ax_sulfidic.taxis.get_major_ticks():
        tick.label1.set_transform(tick.label1.get_transform() + offset_t)

cax = fig.add_axes([1.01, 0.425, 0.020, 0.15])
colorbar = fig.colorbar(cs, cax=cax)
colorbar.set_label("Normalized KDE density", rotation=270, labelpad=14)
colorbar.set_ticks(levels)

plt.show()